In [53]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io

In [54]:
!pip install mlflow albumentations -q

In [55]:
import mlflow
import mlflow.pytorch

In [56]:
!git clone https://github.com/2245-RN-ITBA/skin-dataset-classification.git
import os
os.chdir("skin-dataset-classification")

Cloning into 'skin-dataset-classification'...
remote: Enumerating objects: 934, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 934 (delta 17), reused 15 (delta 15), pack-reused 911 (from 2)
Receiving objects: 100% (934/934), 222.43 MiB | 15.95 MiB/s, done.
Resolving deltas: 100% (31/31), done.
Updating files: 100% (887/887), done.


In [57]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("MLP_Clasificador_Imagenes")

<Experiment: artifact_location='/content/skin-dataset-classification/skin-dataset-classification/mlruns/1', creation_time=1780438597784, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780438597784, lifecycle_stage='active', name='MLP_Clasificador_Imagenes', tags={}, trace_location=None, workspace='default'>

In [58]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [59]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)


In [60]:
# Función para matriz de confusión y clasificación
def log_classification_report(model, loader, writer, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.label_encoder.classes_)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=train_dataset.label_encoder.classes_)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)


In [61]:
# Crear directorio de logs
log_dir = "runs/mlp_experimento_1"
writer = SummaryWriter(log_dir=log_dir)

In [62]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.image_paths = []
        self.labels = []

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)

        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label


In [63]:
train_transform = A.Compose([
    A.Resize(64, 64),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(),
    ToTensorV2()
])


In [64]:
val_test_transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(),
    ToTensorV2()
])


In [65]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [66]:
train_dataset = CustomImageDataset(train_dir, transform=train_transform)
val_dataset   = CustomImageDataset(val_dir, transform=val_test_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)


# PRIMER CAMBIO

## Arquitectura del modelo

Se define la clase `MLPClassifier` con soporte para:
- **Dropout** entre capas (regularización)
- **BatchNorm1d** después de cada capa lineal
- **Inicialización de pesos** configurable (He, Xavier, uniforme)

Esto permite comparar distintas configuraciones cambiando los parámetros al instanciar el modelo.

In [67]:
# MODIFICADA

class MLPClassifier(nn.Module):
    """
    MLP con soporte para Dropout, BatchNorm e inicialización configurable.

    Parámetros
    ----------
    input_size   : int   - tamaño de entrada aplanada (64*64*3 por defecto)
    num_classes  : int   - número de clases de salida
    dropout_p    : float - probabilidad de Dropout (0.0 = sin Dropout)
    use_batchnorm: bool  - si True agrega BatchNorm1d después de cada Linear
    init_mode    : str   - 'he' | 'xavier' | 'uniform' | 'default'
    """
    def __init__(self,
                 input_size=64*64*3,
                 num_classes=10,
                 dropout_p=0.0,
                 use_batchnorm=False,
                 init_mode='default'):
        super().__init__()

        layers = [nn.Flatten()]

        # Bloque 1: Linear -> (BN) -> ReLU -> (Dropout)
        layers.append(nn.Linear(input_size, 512))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(512))
        layers.append(nn.ReLU())
        if dropout_p > 0.0:
            layers.append(nn.Dropout(p=dropout_p))

        # Bloque 2: Linear -> (BN) -> ReLU -> (Dropout)
        layers.append(nn.Linear(512, 128))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU())
        if dropout_p > 0.0:
            layers.append(nn.Dropout(p=dropout_p))

        # Capa de salida
        layers.append(nn.Linear(128, num_classes))

        self.model = nn.Sequential(*layers)

        # Inicialización de pesos
        if init_mode != 'default':
            self.init_weights(init_mode)

    def init_weights(self, mode='he'):
        """Inicializa los pesos de las capas Linear."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if mode == 'he':
                    nn.init.kaiming_normal_(m.weight)
                elif mode == 'xavier':
                    nn.init.xavier_uniform_(m.weight)
                elif mode == 'uniform':
                    nn.init.uniform_(m.weight, -0.1, 0.1)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)

In [68]:
# MODIFICADA

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train_dataset.labels))

# Configuración del experimento:
# Cambiar dropout_p, use_batchnorm e init_mode para probar distintas combinaciones
model = MLPClassifier(
    num_classes=num_classes,
    dropout_p=0.5,          # 0.0 = sin Dropout
    use_batchnorm=True,     # False = sin BatchNorm
    init_mode='he'          # 'he' | 'xavier' | 'uniform' | 'default'
).to(device)

criterion = nn.CrossEntropyLoss()

# Weight Decay = regularización L2 sobre los pesos
# Cambiar weight_decay=0.0 para comparar sin L2
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"Modelo en: {device}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Modelo en: cpu
Parámetros entrenables: 6,360,073


In [69]:
# Entrenamiento y validación
def evaluate(model, loader, epoch=None, prefix="val"):
    log_classification_report(model, val_loader, writer, step=epoch, prefix="val")
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

## Loop de entrenamiento

Además de las métricas de loss y accuracy, se loguean:
- **Histogramas de pesos** por época (`writer.add_histogram`) → permite observar la distribución de los parámetros y detectar problemas de inicialización.
- Todos los hiperparámetros del experimento en MLflow para facilitar la comparación.

In [70]:
# Loop de entrenamiento
n_epochs = 10
with mlflow.start_run():
    # Log hiperparámetros
    mlflow.log_params({
        "model": "MLPClassifier",
        "input_size": 64*64*3,
        "batch_size": batch_size,
        "lr": 1e-3,
        "epochs": n_epochs,
        "optimizer": "Adam",
        "loss_fn": "CrossEntropyLoss",
        "train_dir": train_dir,
        "val_dir": val_dir,
    })
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total
        val_loss, val_acc = evaluate(model, val_loader, epoch=epoch, prefix="val")

        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

        writer.add_scalar("train/loss", train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc, epoch)


    #  Histogramas de pesos
        for name, param in model.named_parameters():
            writer.add_histogram(name, param, epoch)


        # Log en MLflow
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        }, step=epoch)

        # Guardar modelo
    torch.save(model.state_dict(), "mlp_model.pth")
    print("Modelo guardado como 'mlp_model.pth'")
    mlflow.log_artifact("mlp_model.pth")
    mlflow.pytorch.log_model(model, artifact_path="pytorch_model")
    print("Modelo guardado como 'mlp_model.pth'")

Epoch 1/10: 100%|██████████| 22/22 [00:19<00:00,  1.15it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

Epoch 1:
  Train Loss: 2.2425, Accuracy: 27.55%
  Val   Loss: 1.8088, Accuracy: 41.99%


Epoch 2/10: 100%|██████████| 22/22 [00:09<00:00,  2.37it/s]


Epoch 2:
  Train Loss: 1.9017, Accuracy: 37.73%
  Val   Loss: 1.5733, Accuracy: 39.23%


Epoch 3/10: 100%|██████████| 22/22 [00:08<00:00,  2.66it/s]


...

Epoch 10:
  Train Loss: 1.3323, Accuracy: 51.79%
  Val   Loss: 1.1635, Accuracy: 53.04%


2026/06/02 22:43:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 22:43:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/02 22:43:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Modelo guardado como 'mlp_model.pth'


2026/06/02 22:43:58 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Modelo guardado como 'mlp_model.pth'


In [71]:
!tensorboard --logdir=runs/mlp_experimento_1


2026-06-02 22:44:05.600851: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.20.0 at http://localhost:6006/ (Press CTRL+C to quit)
